In [ ]:
# IMPORTS
from egomimic.rldb.utils import S3RLDBDataset
import torch
import numpy as np
from egomimic.utils.egomimicUtils import CameraTransforms, draw_actions
import torchvision.io as io
import os
from egomimic.algo.hpt import HPT
from egomimic.utils.egomimicUtils import nds
import mediapy as media

## Aria Data

In [ ]:
dataset = S3RLDBDataset(
    filters={"episode_hash": "2026-01-22-01-10-22-624000"}, mode="total", cache_root="/coc/flash7/skareer6/CacheEgoVerse/.cache", embodiment="aria_bimanual"
)

In [ ]:
# Make data_loader
data_loader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False)

In [ ]:
camera_transforms = CameraTransforms(intrinsics_key="base", extrinsics_key=None)

In [ ]:
def visualize_actions(ims, actions, extrinsics, intrinsics, arm="both"):
    for b in range(ims.shape[0]):
        if actions.shape[-1] == 7 or actions.shape[-1] == 14:
            ac_type = "joints"
        elif actions.shape[-1] == 3 or actions.shape[-1] == 6:
            ac_type = "xyz"
        else:
            raise ValueError(f"Unknown action type with shape {actions.shape}")

        ims[b] = draw_actions(
            ims[b], ac_type, "Purples", actions[b], extrinsics, intrinsics, arm=arm
        )

    return ims

In [ ]:
ims = []
image_key = "observations.images.front_img_1"
actions_key = "actions_cartesian"
for i, data in enumerate(data_loader):
    im = data[image_key][0].permute(1, 2, 0).cpu().numpy() * 255
    im = im.astype(np.uint8)
    actions = data[actions_key][0].cpu().numpy()
    xyz, rot = HPT._extract_xyz(torch.from_numpy(actions))
    im = draw_actions(im, "xyz", "Purples", xyz.numpy(), None, camera_transforms.intrinsics, arm="both")
    ims.append(im)
    if i > 200:
        break

media.show_video(ims, fps=30)

## Eva Data

In [ ]:
dataset = S3RLDBDataset(
    filters={"episode_hash": "2026-01-26-20-58-18-408000"}, mode="total", cache_root="/coc/flash7/skareer6/CacheEgoVerse/.cache", embodiment="eva_bimanual"
)

In [ ]:
# Make data_loader
data_loader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False)

In [ ]:
ims = []
image_key = "observations.images.front_img_1"
actions_key = "actions_cartesian"
for i, data in enumerate(data_loader):
    im = data[image_key][0].permute(1, 2, 0).cpu().numpy() * 255
    im = im.astype(np.uint8)
    actions = data[actions_key][0].cpu().numpy()
    xyz, rot = HPT._extract_xyz(torch.from_numpy(actions))
    im = draw_actions(im, "xyz", "Purples", xyz.numpy(), None, camera_transforms.intrinsics, arm="both")
    ims.append(im)
    if i > 500:
        break

media.show_video(ims, fps=30)